# Topology PoC — интерактивный запуск

Каждая ячейка — один логический блок. Запускай последовательно или отдельно для отладки.

**Режимы:**
- `STUB = True` — заглушки вместо LLM, бесплатно, только для проверки пайплайна
- `STUB = False` — реальные вызовы OpenAI API

In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# ищем .env от корня проекта вверх
ROOT = Path(".").resolve()
for p in [ROOT, ROOT.parent]:
    env_file = p / ".env"
    if env_file.exists():
        load_dotenv(env_file, override=True)
        print(f".env loaded from {env_file}")
        break
else:
    print("WARNING: .env not found")

print(f"OPENAI_API_KEY set: {bool(os.environ.get('OPENAI_API_KEY'))}")


## 0. Конфигурация

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL       = "gpt-3.5-turbo"
DATASET     = "gsm8k"   # ← "gsm8k" или "math500"
N_QUESTIONS = 5
N_AGENTS    = 3
STUB        = True      # ← True = без API, False = реальные вызовы

print(f"ROOT: {ROOT}")
print(f"stub={STUB}, model={MODEL}, dataset={DATASET}, n_questions={N_QUESTIONS}, n_agents={N_AGENTS}")

## 1. Загрузка датасета

Поддерживаются `gsm8k` (целочисленные ответы) и `math500` (LaTeX-выражения).
Переменная `DATASET` задаётся в ячейке конфигурации.

In [ ]:
import re
from datasets import load_dataset

def load_questions(dataset: str, n: int) -> list:
    if dataset == "gsm8k":
        try:
            ds = load_dataset("openai/gsm8k", "main", split=f"test[:{n}]")
        except Exception:
            import urllib.request, json as _j
            url = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"
            print("  HuggingFace недоступен, качаю с GitHub...")
            with urllib.request.urlopen(url) as r:
                lines = r.read().decode().splitlines()
            ds = [_j.loads(l) for l in lines[:n]]
        return [
            {
                "question": item["question"],
                "answer":   re.search(r"####\s*(-?\d[\d,]*)", item["answer"]).group(1).replace(",", ""),
            }
            for item in ds
        ]
    if dataset == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split=f"test[:{n}]")
        return [
            {
                "question": item["problem"],
                "answer":   item["answer"],
                "subject":  item.get("subject", ""),
                "level":    item.get("level", ""),
            }
            for item in ds
        ]
    raise ValueError(f"Unknown dataset: {dataset!r}")


def _normalize(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\\left\s*[\(\[\{]", "", s)
    s = re.sub(r"\\right\s*[\)\]\}]", "", s)
    s = re.sub(r"\\frac\{([^}]+)\}\{([^}]+)\}", r"\1/\2", s)
    s = re.sub(r"[$\\]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

def _to_float(s):
    try:
        if "/" in s and s.count("/") == 1:
            a, b = s.split("/")
            return float(a) / float(b)
        return float(s)
    except Exception:
        return None

def answers_match(pred_raw, gt: str) -> bool:
    if pred_raw is None:
        return False
    p, g = _normalize(pred_raw), _normalize(gt)
    if p == g:
        return True
    pf, gf = _to_float(p), _to_float(g)
    if pf is not None and gf is not None:
        return abs(pf - gf) < 1e-6
    return False


questions = load_questions(DATASET, N_QUESTIONS)

print(f"Датасет  : {DATASET}")
print(f"Загружено: {len(questions)} задач")
print(f"\nПример вопроса:\n{questions[0]['question'][:150]}...")
print(f"\nGT ответ : {questions[0]['answer']}")

## 2. Топологии — построение и визуализация

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from topologies.definitions import get_topologies

topologies = get_topologies(n_agents=N_AGENTS)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.suptitle("Топологии MAS", fontsize=13)

NODE_COLORS = {"task": "#94a3b8", "agent": "#3b82f6"}

for ax, (name, G) in zip(axes, topologies.items()):
    pos = nx.spring_layout(G, seed=42)
    colors = [NODE_COLORS["task"] if n == "task" else NODE_COLORS["agent"] for n in G.nodes]
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=700, font_size=8, font_color="white",
            arrows=True, arrowsize=15, edge_color="#64748b")
    ax.set_title(f"{name}\n{G.number_of_nodes()} nodes, {G.number_of_edges()} edges", fontsize=10)

plt.tight_layout()
plt.savefig(ROOT / "results" / "topologies.png", dpi=120, bbox_inches="tight")
plt.show()

for name, G in topologies.items():
    order = list(nx.topological_sort(G))
    print(f"{name:8s} | nodes: {list(G.nodes)} | topo_order: {order}")

## 3. Метрики графов

In [ ]:
import pandas as pd
from metrics.graph_metrics import TopologyMetrics

engine = TopologyMetrics()
metrics_rows = []

for name, G in topologies.items():
    m = engine.compute(G, name)
    metrics_rows.append({
        "topology": name,
        "diameter": round(m.diameter, 3),
        "avg_degree": round(m.avg_degree, 3),
        "structural_entropy": round(m.structural_entropy, 3),
        "spectral_gap": round(m.spectral_gap, 3),
        "task_centrality": round(m.task_centrality, 3),
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("topology")
metrics_df

## 4. Инициализация агентов

In [ ]:
from mas.agent import Agent, AgentConfig
from mas.runner import MASRunner
from mas.prompts import assign_roles, SYSTEM_PROMPTS

_KNOWN_ROLES  = set(SYSTEM_PROMPTS.keys())
_ROLES_CYCLE  = ["solver", "critic", "aggregator"]

def _infer_role(node: str, index: int) -> str:
    if node == "agg":
        return "aggregator"
    if node in _KNOWN_ROLES:
        return node
    for role in _KNOWN_ROLES:
        if node.startswith(role):
            return role
    return _ROLES_CYCLE[index % len(_ROLES_CYCLE)]

def build_agents(graph: nx.DiGraph) -> dict:
    from topologies.definitions import HYBRID_ROLES
    order      = [n for n in nx.topological_sort(graph) if n != "task"]
    hybrid_map = HYBRID_ROLES.get(getattr(graph, "_name", ""), {})
    return {
        node: Agent(AgentConfig(
            agent_id=node,
            role=hybrid_map.get(node) or _infer_role(node, order.index(node)),
            model=MODEL,
            stub=STUB,
        ))
        for node in order
    }

print("STUB =", STUB)
test_agent = Agent(AgentConfig("A0", "solver", stub=True))
print("Smoke-test агента:", test_agent.run("What is 2 + 2?", []))

## 5. Smoke-test одной топологии на одной задаче

Запусти эту ячейку чтобы убедиться что пайплайн проходит насквозь до сохранения ответа.

In [ ]:
SMOKE_TOPO = "chain"   # ← chain / star / full / random
SMOKE_Q    = 0         # ← индекс задачи из датасета

G_smoke  = topologies[SMOKE_TOPO]
agents   = build_agents(G_smoke)
runner   = MASRunner(G_smoke, agents)

q   = questions[SMOKE_Q]
out = runner.run(q["question"])

from mas.prompts import parse_answer_str
pred = parse_answer_str(out)
ok   = answers_match(pred, q["answer"])

print(f"Топология : {SMOKE_TOPO}")
print(f"Датасет   : {DATASET}")
print(f"Вопрос    : {q['question'][:100]}...")
print(f"GT ответ  : {q['answer']}")
print(f"\nОтвет MAS (полный):\n{out}")
print(f"\nПарсинг   : {pred!r}")
print(f"Верно     : {ok}")

## 6. Полный прогон — все топологии × все задачи

In [ ]:
import time
from mas.prompts import parse_answer_str

results = []

for topo_name, G in topologies.items():
    agents  = build_agents(G)
    runner  = MASRunner(G, agents)
    m       = engine.compute(G, topo_name)
    t0      = time.perf_counter()

    correct, total = 0, len(questions)
    per_question   = []

    for i, item in enumerate(questions):
        out  = runner.run(item["question"])
        pred = parse_answer_str(out)
        ok   = answers_match(pred, item["answer"])
        if ok:
            correct += 1
        per_question.append({"q": i, "gt": item["answer"], "pred": pred, "ok": ok})

    acc      = correct / total
    duration = round(time.perf_counter() - t0, 2)
    print(f"{topo_name:8s}  accuracy={acc:.2f}  ({correct}/{total})  {duration}s")

    results.append({
        "topology":     topo_name,
        "accuracy":     acc,
        "duration_sec": duration,
        "per_question": per_question,
        "metrics": {
            "diameter":           m.diameter,
            "avg_degree":         m.avg_degree,
            "structural_entropy": m.structural_entropy,
            "spectral_gap":       m.spectral_gap,
            "task_centrality":    m.task_centrality,
        },
    })

print("\nГотово.")

## 7. Сводная таблица результатов

In [ ]:
summary_rows = []
for r in results:
    summary_rows.append({
        "topology": r["topology"],
        "accuracy": r["accuracy"],
        **{k: round(v, 3) for k, v in r["metrics"].items()},
    })

summary_df = pd.DataFrame(summary_rows).set_index("topology").sort_values("accuracy", ascending=False)
summary_df

## 8. Разбор ошибок — какие задачи провалились

In [ ]:
INSPECT_TOPO = "chain"   # ← какую топологию разбираем

topo_result = next(r for r in results if r["topology"] == INSPECT_TOPO)

errors = [pq for pq in topo_result["per_question"] if not pq["ok"]]
print(f"Ошибки [{INSPECT_TOPO}]: {len(errors)}/{len(topo_result['per_question'])}")
for e in errors:
    q_text = questions[e["q"]]["question"][:80]
    print(f"  Q{e['q']}: GT={e['gt']}, pred={e['pred']} | {q_text}...")

## 9. Сохранение results.json

In [ ]:
import json
from datetime import datetime, timezone

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)

run_ts = datetime.now(timezone.utc)
run_id = run_ts.strftime("%Y%m%d_%H%M%S")

slim = [{k: v for k, v in r.items() if k != "per_question"} for r in results]

run_doc = {
    "meta": {
        "run_id":       run_id,
        "timestamp":    run_ts.isoformat(),
        "model":        MODEL,
        "dataset":      DATASET,
        "n_questions":  len(questions),
        "stub":         STUB,
        "n_topologies": len(topologies),
        "mean_accuracy": round(sum(r["accuracy"] for r in results) / len(results), 4),
    },
    "results": slim,
}

run_file = out_dir / f"run_{run_id}.json"
run_file.write_text(json.dumps(run_doc, indent=2, ensure_ascii=False))
(out_dir / "results.json").write_text(json.dumps(run_doc, indent=2, ensure_ascii=False))

# обновляем индекс
index_path = out_dir / "runs_index.json"
index = json.loads(index_path.read_text()) if index_path.exists() else []
index.append({**run_doc["meta"], "results_file": str(run_file)})
index_path.write_text(json.dumps(index, indent=2, ensure_ascii=False))

print(f"Run ID  : {run_id}")
print(f"Сохранено → {run_file}")
print(f"Индекс  → {index_path}")
print(json.dumps(run_doc["meta"], indent=2, ensure_ascii=False))